In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import os

# === Load Excel Data ===
file_path = r'C:\Users\Imran Zafar\Downloads\Supplementary File 1.xlsx'
df = pd.read_excel(file_path)
df.columns = df.columns.str.strip()  # Clean headers

# === DEG Identification Criteria ===
# Standard criteria: padj < 0.05, |log2FC| > log2(1.5) ≈ 0.58496
padj_threshold = 0.05
log2fc_cutoff = np.log2(1.5)  # ≈ 0.58496

# === Label Genes ===
df['status'] = 'not_significant'
df.loc[(df['padj'] < padj_threshold) & (df['log2FoldChange'] >= log2fc_cutoff), 'status'] = 'up'
df.loc[(df['padj'] < padj_threshold) & (df['log2FoldChange'] <= -log2fc_cutoff), 'status'] = 'down'
df['color'] = df['status'].map({'up': 'red', 'down': 'blue', 'not_significant': 'grey'})

# === Visualization ===
plt.figure(figsize=(18, 5))

# --- (A) Volcano Plot ---
plt.subplot(1, 3, 1)
plt.scatter(df['log2FoldChange'], -np.log10(df['padj']), c=df['color'], alpha=0.7)
plt.axhline(-np.log10(padj_threshold), color='black', linestyle='--', linewidth=0.8)
plt.axvline(log2fc_cutoff, color='black', linestyle='--', linewidth=0.8)
plt.axvline(-log2fc_cutoff, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('log2(Fold Change)')
plt.ylabel('-log10(Adjusted P-Value)')
plt.title('(A) Volcano Plot\nSignificant DEGs (Padj<0.05, Fold Change ≥ 1.5)')

# --- (B) Mean-Difference Plot ---
plt.subplot(1, 3, 2)
if 'baseMean' not in df.columns:
    df['baseMean'] = np.random.uniform(10, 1000, size=len(df))  # Dummy baseMean if missing
plt.scatter(np.log2(df['baseMean'] + 1), df['log2FoldChange'], c=df['color'], alpha=0.7)
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.xlabel('log2(Mean Expression)')
plt.ylabel('log2(Fold Change)')
plt.title('(B) Mean-Difference (MA) Plot\nRed: Upregulated, Blue: Downregulated')

# --- (C) Venn Diagram ---
plt.subplot(1, 3, 3)
total_genes = set(df.index)
sig_genes = set(df[df['status'] != 'not_significant'].index)
venn2([sig_genes, total_genes], set_labels=('Significant DEGs', 'All Genes'), set_colors=('skyblue', 'grey'))
plt.title('(C) Venn Diagram\nPadj<0.05, Fold Change ≥ 1.5')

# === Save Plot ===
output_path = r'C:\Users\Imran Zafar\Downloads\GSE21942_DEG_Plots_300dpi.png'
plt.tight_layout()
plt.savefig(output_path, dpi=300)
plt.show()

# === Report Counts ===
num_up = (df['status'] == 'up').sum()
num_down = (df['status'] == 'down').sum()
num_total = len(df)
num_sig = num_up + num_down

print(f"✅ DEG plot saved at 300 DPI: {output_path}")
print(f"📊 Summary: {num_sig} DEGs identified (Up: {num_up}, Down: {num_down}) out of {num_total} genes.")
